In [1]:
# Statistics

In [2]:
# Import relevant Python packages:
import xarray as xr
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

# Following pip installation as shown on the LT Toolbox github:
import lt_toolbox as ltt
import gsw.density as density

# Searching the OceanDataCatalog dataset

from OceanDataStore import OceanDataCatalog

catalog = OceanDataCatalog(catalog_name="noc-model-stac")

#catalog.available_collections

#Era 5 probs the best here

#Searching the ERA5 collection
#catalog.search(collection='noc-npd-era5')
#catalog.available_items

t1 = catalog.open_dataset(id= 'noc-npd-era5/npd-eorca1-era5v1/gn/T1y',
                          start_datetime='2000-01',
                          end_datetime='2010-12',
                          )

#t1


d1 = catalog.open_dataset(id= 'noc-npd-era5/npd-eorca1-era5v1/gn/domain/domain_cfg',
                          )

d1

  2026-04-20T13:28:45.335058Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error(Connect, HttpTimeoutError { kind: "HTTP connect", duration: 1s }), connection: Unknown } }) }))
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.12/src/imds/region.rs:66

  2026-04-20T13:28:52.829462Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error

<xarray.Dataset> Size: 710MB
Dimensions:       (y: 331, x: 360, nav_lev: 75)
Coordinates:
  * y             (y) int64 3kB 0 1 2 3 4 5 6 7 ... 324 325 326 327 328 329 330
  * x             (x) int64 3kB 0 1 2 3 4 5 6 7 ... 353 354 355 356 357 358 359
  * nav_lev       (nav_lev) int64 600B 0 1 2 3 4 5 6 7 ... 68 69 70 71 72 73 74
Data variables: (12/49)
    atlmsk        (y, x) float32 477kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    bottom_level  (y, x) int32 477kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    e1f           (y, x) float64 953kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    e1t           (y, x) float64 953kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    e2t           (y, x) float64 953kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    bathy_metry   (y, x) float32 477kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    ...            ...
    tmask         (nav_lev, y, x) int8 9MB dask.array<chunksize=(75, 331, 360), meta=np.ndarray>
    misf          (y, x) int32 477kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    umaskutil     (y, x) int8 119kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    umask         (nav_lev, y, x) int8 9MB dask.array<chunksize=(75, 331, 360), meta=np.ndarray>
    wmask         (nav_lev, y, x) bool 9MB dask.array<chunksize=(75, 331, 360), meta=np.ndarray>
    vmaskutil     (y, x) int8 119kB dask.array<chunksize=(331, 360), meta=np.ndarray>
Attributes:
    CfgName:    UNKNOWN
    CfgIndex:   -999
    Iperio:     1
    Jperio:     0
    NFold:      1
    NFtype:     F
    VertCoord:  zps
    IsfCav:     0
    file_name:  mesh_mask.nc
    TimeStamp:  01/03/2025 22:19:49 -0000

In [4]:
# Creating parquet files - Only needs to be done once
import polars as pl
import lt_toolbox as ltt
from pathlib import Path
import gc

# Model grid
lon_mdl = t1.nav_lon.values
lat_mdl = t1.nav_lat.values
depth_mdl = t1.deptht.values

# Output directory
output_dir = Path("/dssgfs01/scratch/emdh1n25/project_1m_RAPID_new/eORCA1/processed_traj")
output_dir.mkdir(parents=True, exist_ok=True)

for year in range(1990, 2000):
    for month in range(1, 13):

        mm = f"{month:02d}"
        key = f"{year}_{mm}"

        traj_filepath = Path(
            f"/dssgfs01/scratch/emdh1n25/project_1m_RAPID/"
            f"eORCA1/eORCA1_output/parquet_new/"
            f"ORCA1_NPD_{mm}_{year}_25_1m_run.parquet"
        )

        output_file = output_dir / f"{key}.parquet"

        if not traj_filepath.exists():
            print(f"Missing {key}, skipping")
            continue

        if output_file.exists():
            print(f"Skipping {key}, already processed")
            continue

        print(f"Processing {key}")

        try:
            # Load
            dataset = pl.read_parquet(traj_filepath)

            # Build trajectory object
            traj = ltt.TrajFrame(source=dataset, condense=True)

            # Convert time
            traj_geo = traj.use_datetime(
                start_date=f"{year}-01-01",
                unit="s"
            )

            # Save original x, y, z
            x_orig = traj_geo.data["x"]
            y_orig = traj_geo.data["y"]
            z_orig = traj_geo.data["z"]

            # Transform coordinates
            traj_geo_interp = traj_geo.transform_trajectory_coords(
                lon=lon_mdl,
                lat=lat_mdl,
                depth=depth_mdl
            )

            # Re-add original indices (overwrite if needed)
            traj_geo_interp.data = traj_geo_interp.data.with_columns([
                x_orig.alias("x"),
                y_orig.alias("y"),
                z_orig.alias("z"),
            ])

            # Reorder columns
            traj_geo_interp.data = traj_geo_interp.data.select(
                ["id", "x", "y", "z"]
                + [c for c in traj_geo_interp.data.columns if c not in ["id", "x", "y", "z"]]
            )

            # Save to disk
            traj_geo_interp.data.write_parquet(output_file)

        except Exception as e:
            print(f"Error processing {key}: {e}")
            continue

        # Cleanup
        del dataset, traj, traj_geo, traj_geo_interp
        gc.collect()

print("Done")

Processing 1990_01
Processing 1990_02
Processing 1990_03
Processing 1990_04
Processing 1990_05
Processing 1990_06
Processing 1990_07
Processing 1990_08
Processing 1990_09
Processing 1990_10
Processing 1990_11
Processing 1990_12
Processing 1991_01
Processing 1991_02
Processing 1991_03
Processing 1991_04
Processing 1991_05
Processing 1991_06
Processing 1991_07
Processing 1991_08
Processing 1991_09
Processing 1991_10
Processing 1991_11
Processing 1991_12
Processing 1992_01
Processing 1992_02
Processing 1992_03
Processing 1992_04
Processing 1992_05
Processing 1992_06
Processing 1992_07
Processing 1992_08
Processing 1992_09
Processing 1992_10
Processing 1992_11
Processing 1992_12
Processing 1993_01
Processing 1993_02
Processing 1993_03
Processing 1993_04
Processing 1993_05
Processing 1993_06
Processing 1993_07
Processing 1993_08
Processing 1993_09
Processing 1993_10
Processing 1993_11
Processing 1993_12
Processing 1994_01
Processing 1994_02
Processing 1994_03
Processing 1994_04
Processing 1